In [ ]:
from pathlib import Path
import hashlib, json, time
import numpy as np
import pandas as pd
import xarray as xr

HW_DIR = Path("../outputs/heatwave")
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")
OUT_BASE = Path("../outputs/simulation")
SPLITS = {
    "main_2016-2019_to_2020-2023": ([2016, 2017, 2018, 2019], [2020, 2021, 2022, 2023]),
    "alt_2016-2018_to_2019-2023": ([2016, 2017, 2018], [2019, 2020, 2021, 2022, 2023]),
}
SCHEME = "local_p90"
DELTA_T_VALUES = [0.5, 1.0, 1.5]
BUDGET_SHARES = [0.10, 0.20, 0.30]
TYPES = ["day", "night", "compound", "any"]
RULES = ["oracle", "hwd_foresight", "population", "density", "height",
         "hwd_hist", "exposure_hist", "response_hist", "random"]
KEY = ["city", "cell_id", "year"]
YEARS = list(range(2016, 2024))
DATES = pd.date_range("2016-01-01", "2025-12-31")
YEAR_ARRAY, DOY = DATES.year.to_numpy(), DATES.dayofyear.to_numpy() - 1

def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def write_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def as_bool(s):
    q = s.astype(str).str.strip().str.lower()
    require(q.isin(["true", "false", "1", "0"]).all(), f"Abnormal boolean field: {s.name}")
    return q.isin(["true", "1"])

def segments(mask):
    edges = np.diff(np.r_[0, np.asarray(mask, dtype=np.int8), 0])
    return zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1))

def count_years(tx, tn, qx, qn):
    valid = np.isfinite(tx) & np.isfinite(tn) & np.isfinite(qx) & np.isfinite(qn)
    hx, hn = valid & (tx >= qx), valid & (tn >= qn)
    candidates = [hx & ~hn, hn & ~hx, hx & hn]
    result = np.empty((len(YEARS), 4), dtype=float)
    for k, candidate in enumerate(candidates):
        flag = np.zeros(len(tx), dtype=float)
        for start, end in segments(candidate):
            if end - start >= 3:
                flag[start:end] = 1
        for start, end in segments(candidate | ~valid):
            if (~valid[start:end]).any():
                flag[start:end] = np.nan
        flag[~valid] = np.nan
        for iy, year in enumerate(YEARS):
            result[iy, k] = flag[YEAR_ARRAY == year].sum()
    result[:, 3] = result[:, :3].sum(axis=1)
    return result

def expected_benefit(scores, response, k):
    scores, response = np.asarray(scores, float), np.asarray(response, float)
    require(np.isfinite(scores).all() and np.isfinite(response).all(), "Ranking score or benefit is missing")
    require(1 <= k <= len(scores), "Abnormal number of sited grid cells")
    cutoff = np.partition(scores, len(scores) - k)[len(scores) - k]
    higher, tied = scores > cutoff, scores == cutoff
    slots = int(k - higher.sum())
    return float(response[higher].sum() + slots * response[tied].mean())

def run_multiyear_fixed():
    t0 = time.time()
    for train, test in SPLITS.values():
        require(not set(train) & set(test) and max(train) < min(test), "Abnormal time split")
        require(set(train + test) == set(YEARS), "This cell requires every split to cover 2016–2023")
    require(all(0 < b <= 1 for b in BUDGET_SHARES), "Budget must be in (0,1]")
    require(all(x > 0 for x in DELTA_T_VALUES), "Cooling magnitude must be positive")
    provenance = {}
    def remember(path, expected=None):
        path = Path(path)
        h = sha256(path)
        require(expected is None or h == expected, f"Input checksum mismatch: {path}")
        provenance[str(path)] = h
    remember(PANEL)
    remember(HW_DIR / "summary.json")
    hs = json.loads((HW_DIR / "summary.json").read_text(encoding="utf-8"))
    manifest_path = Path(hs["input_manifest"])
    remember(manifest_path, hs["input_manifest_sha256"])
    manifest = pd.read_csv(manifest_path)
    require(not manifest.city.duplicated().any(), "Duplicate cities in the temperature manifest")
    manifest = manifest.set_index("city")
    p = pd.read_csv(PANEL)
    require(len(p) == 5216 and not p.duplicated(KEY).any(), "Abnormal panel row count or keys")
    require(p.city.nunique() == 35 and set(p.year) == set(YEARS), "Abnormal city or year")
    require(p.scheme.eq(SCHEME).all() and p.pct.eq("p90").all(), "Not the local P90 main panel")
    for c in ["heatwave_annual_complete", "eligible_density_model", "eligible_height_model"]:
        p[c] = as_bool(p[c])
    cities = sorted(p.city.unique())
    require(set(cities) <= set(manifest.index), "Cities missing from the temperature manifest")
    out = OUT_BASE
    (out / "by_city").mkdir(parents=True, exist_ok=True)
    write_json(out / "configuration.json", dict(panel=str(PANEL), heatwave_dir=str(HW_DIR), scheme=SCHEME,
        splits={k: dict(train=v[0], test=v[1]) for k, v in SPLITS.items()},
        delta_t=DELTA_T_VALUES, budgets=BUDGET_SHARES))
    write_json(out / "summary.json", dict(status="running"))
    all_eff = []
    for ic, city in enumerate(cities, 1):
        pc = p[p.city.eq(city)].copy()
        nc = manifest_path.parent / manifest.loc[city, "output_nc"]
        thpath = HW_DIR / "cities" / city / "thresholds.nc"
        remember(nc, manifest.loc[city, "output_sha256"])
        remember(thpath)
        with xr.open_dataset(nc) as ds:
            require(pd.DatetimeIndex(ds.time.values).equals(DATES), f"{city} abnormal date axis")
            cell_ids = ds.cell_id.values.astype(int)
            tx = ds.tx_C.transpose("time", "cell_id").values.astype(float)
            tn = ds.tn_C.transpose("time", "cell_id").values.astype(float)
        require(len(set(cell_ids)) == len(cell_ids), f"{city} duplicate temperature grid cells")
        with xr.open_dataset(thpath) as ds:
            th = ds.sel(cell_id=cell_ids, scheme=SCHEME)
            require(np.array_equal(th.doy.values, np.arange(1, 367)), "Abnormal threshold calendar axis")
            qx = th.threshold_tx_C.transpose("doy", "cell_id").values.astype(float)[DOY]
            qn = th.threshold_tn_C.transpose("doy", "cell_id").values.astype(float)[DOY]
        index = {int(c): j for j, c in enumerate(cell_ids)}
        city_eff, city_cells = [], []
        for split, (train, test) in SPLITS.items():
            ti, ei = [YEARS.index(y) for y in train], [YEARS.index(y) for y in test]
            eligible = []
            for c, g in pc.groupby("cell_id"):
                gh = g[g.year.isin(train)]
                b_ok = gh.eligible_density_model & np.isfinite(gh.bld_coverage) & gh.bld_coverage.gt(0)
                if (len(g) == len(YEARS) and g.heatwave_annual_complete.all()
                    and np.isfinite(g.pop_study).all() and g.pop_study.gt(0).all() and b_ok.any()):
                    eligible.append(int(c))
            eligible = sorted(eligible)
            n = len(eligible)
            if n < 3:
                continue
            require(set(eligible) <= set(index), f"{city} temperature data lack candidate grid cells")
            pop = pc.pivot(index="year", columns="cell_id", values="pop_study").loc[YEARS, eligible].to_numpy(float)
            gh = pc[pc.year.isin(train)].copy()
            good_b = gh.eligible_density_model & np.isfinite(gh.bld_coverage) & gh.bld_coverage.gt(0)
            density = gh[good_b].groupby("cell_id").bld_coverage.mean().reindex(eligible).to_numpy(float)
            good_h = gh.eligible_height_model & np.isfinite(gh.bld_mean_h_m) & gh.bld_mean_h_m.gt(0)
            height_raw = gh[good_h].groupby("cell_id").bld_mean_h_m.mean().reindex(eligible).to_numpy(float)
            require(np.isnan(height_raw).sum() < n, f"{city} {split}: no candidate grid cell has a valid building height")
            height = np.where(np.isnan(height_raw), -1.0, height_raw)
            base = np.empty((len(YEARS), n, 4))
            for j, c in enumerate(eligible):
                k = index[c]
                base[:, j] = count_years(tx[:, k], tn[:, k], qx[:, k], qn[:, k])
                old = pc[pc.cell_id.eq(c)].set_index("year").loc[YEARS, [f"HW_days_{t}" for t in TYPES[:3]]].to_numpy(float)
                require(np.isfinite(base[:, j]).all() and np.array_equal(base[:, j, :3], old), f"{city} {c} baseline HWD does not match the panel")
            for delta in DELTA_T_VALUES:
                response, historical_response = np.zeros((n, 4)), np.zeros((n, 4))
                for j, c in enumerate(eligible):
                    k = index[c]
                    x, z = tx[:, k].copy(), tn[:, k].copy()
                    mask = np.isin(YEAR_ARRAY, test)
                    x[mask] -= delta; z[mask] -= delta
                    post = count_years(x, z, qx[:, k], qn[:, k])
                    response[j] = (pop[ei, j, None]*(base[ei, j]-post[ei])).sum(0)
                    for iy in ti:
                        x, z = tx[:, k].copy(), tn[:, k].copy()
                        mask = YEAR_ARRAY == YEARS[iy]
                        x[mask] -= delta; z[mask] -= delta
                        post = count_years(x, z, qx[:, k], qn[:, k])
                        historical_response[j] += pop[iy, j]*(base[iy, j]-post[iy])/len(ti)
                require(np.isfinite(response).all() and np.isfinite(historical_response).all(), f"{city} simulated response missing")
                require((response[:, 2] >= -1e-6).all(), "Abnormal cooling response of compound heatwaves")
                require(np.allclose(response[:, 3], response[:, :3].sum(1)), "Abnormal sum of net changes over the three types")
                for it, heat in enumerate(TYPES):
                    r = response[:, it]
                    scores = dict(population=pop[ti].mean(0), density=density, height=height,
                        hwd_hist=base[ti, :, it].mean(0), exposure_hist=(pop[ti]*base[ti, :, it]).mean(0),
                        response_hist=historical_response[:, it], hwd_foresight=base[ei, :, it].mean(0),
                        oracle=r, random=np.zeros(n))
                    baseline = float((pop[ei]*base[ei, :, it]).sum())
                    common = dict(city=city, split=split, heatwave_type=heat, cooling_mode="tmax_tmin",
                        delta_t_C=delta, n_eligible_cells=n, baseline_exposure_test=baseline)
                    for j, c in enumerate(eligible):
                        city_cells.append(common | dict(cell_id=c, response_test=float(r[j]),
                            **{f"score_{s}": float(scores[s][j]) for s in RULES}))
                    for budget in BUDGET_SHARES:
                        ksel = max(1, int(np.ceil(budget*n)))
                        oracle = float(np.sort(r)[-ksel:].sum())
                        info = common | dict(budget_share=budget, n_treated=ksel,
                            actual_cell_share=ksel/n, oracle_avoided=oracle)
                        for rule in RULES:
                            benefit = expected_benefit(scores[rule], r, ksel)
                            require(benefit <= oracle + 1e-6 + abs(oracle)*1e-10, "Rule benefit exceeds oracle")
                            city_eff.append(info | dict(avoided_exposure=benefit, strategy=rule,
                                avoided_person_days_per_year=benefit/len(test),
                                relative_efficiency=benefit/oracle if oracle > 0 else np.nan,
                                avoided_share_of_baseline=benefit/baseline if baseline > 0 else np.nan))
        pd.DataFrame(city_cells).to_csv(out/"by_city"/f"{city}_cell_responses.csv", index=False)
        all_eff.extend(city_eff)
        print(f"[{ic}/{len(cities)}] {city}: {len(city_eff)} results", flush=True)
    eff = pd.DataFrame(all_eff)
    require(not eff.empty, "No usable simulation results")
    case = ["city", "split", "heatwave_type", "cooling_mode", "delta_t_C", "budget_share"]
    require(not eff.duplicated(case+["strategy"]).any(), "Duplicate keys in simulation results")
    require(eff.groupby(case).strategy.nunique().eq(len(RULES)).all(), "Some scenarios lack siting rules")
    require(np.allclose(eff.loc[eff.strategy.eq("oracle"), "avoided_exposure"], eff.loc[eff.strategy.eq("oracle"), "oracle_avoided"]), "oracle check failed")
    eff.to_csv(out/"multiyear_siting_efficiency.csv", index=False)
    for path, h in provenance.items():
        require(sha256(path) == h, f"Input changed during the run: {path}")
    pd.DataFrame([dict(path=k, sha256=v) for k, v in provenance.items()]).to_csv(out/"input_manifest.csv", index=False)
    write_json(out/"summary.json", dict(status="completed", cities_with_results=int(eff.city.nunique()),
        rows=len(eff), runtime_minutes=(time.time()-t0)/60))
    print("Output folder:", out)
    return eff

MULTI_FIXED_EFF = run_multiyear_fixed()